In [15]:
import pandas as pd
import torch
from datasets import load_dataset

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [8]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
    print(f"Device index: {torch.cuda.current_device()}")
else:
    print("CUDA is not available. Using CPU.")
    device = torch.device("cpu")

print(f"Device: {device}")

Using device: NVIDIA GeForce RTX 3060 Ti
Device index: 0
Device: cuda


In [9]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

In [10]:
def classify(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_class_id = logits.argmax().item()
    result = model.config.id2label[predicted_class_id].lower()

    return result

In [11]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1066 entries, 0 to 1065
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1066 non-null   object
 1   label   1066 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 16.8+ KB


In [12]:
test['label'] = test['label'].apply(lambda x: 'positive' if x == 1 else 'negative')

labels = test['label'].unique()
labels

array(['positive', 'negative'], dtype=object)

In [13]:
test['predicted'] = test['text'].apply(classify)

In [16]:
# get metrics
accuracy = accuracy_score(test['label'], test['predicted'])
f1 = f1_score(test['label'], test['predicted'], labels=labels, average='weighted')
recall = recall_score(test['label'], test['predicted'], labels=labels, average='weighted')
precision = precision_score(test['label'], test['predicted'], labels=labels, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"F1: {f1}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")

Accuracy: 0.8968105065666041
F1: 0.896807237397916
Recall: 0.8968105065666041
Precision: 0.8968607971047656
